In [ ]:
# نسخه‌ی کاملاً رایگان — مدل embedding به‌صورت محلی روی CPU اجرا می‌شود (هیچ API و کلیدی لازم نیست)
# خط اول ویلِ سبک CPUِ torch را نصب می‌کند (وگرنه pip بیلد CUDA حدوداً 2.5GB را می‌کشد)
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
!pip install -q sentence-transformers "psycopg[binary]" pgvector pypdf tiktoken numpy python-dotenv fonttools

In [ ]:
import os
from urllib.parse import urlparse

try:
    from dotenv import load_dotenv
    _env_file_found = bool(load_dotenv())
except ImportError:
    _env_file_found = False

DB_URI = os.environ.get("DATABASE_URL2")

if not DB_URI:
    raise ValueError(
        "❌ DATABASE_URL2 پیدا نشد.\n"
        "   راه ۱: کنار این نوت‌بوک فایل .env بسازید (الگو: env.example) و سلول را دوباره اجرا کنید.\n"
        "   راه ۲: متغیر را قبل از اجرای jupyter در ترمینال ست کنید (export / set).\n"
    )

print("✅ Secrets loaded"
      + (" از فایل .env" if _env_file_found else " از متغیرهای محیطی سیستم (فایل .env پیدا نشد)"))
print(f"   Supabase host : {urlparse(DB_URI).hostname}")

## 🆓 Embedding کاملاً رایگان — مدل محلی روی CPU لپ‌تاپ

هیچ API و کلید و سقف ماهانه‌ای در کار نیست: همان مدل **رایگانِ متن‌باز** `BAAI/bge-large-en-v1.5`
که در نسخه‌های قبلی از طریق API سرو می‌شد، این‌بار مستقیم روی CPU لپ‌تاپ اجرا می‌شود.

- بعدِ بردار همان **1024** → ستون `vector(1024)` جدول Supabase بدون تغییر.
- مدل دقیقاً همان است → بردارهای جدید با هر بردار قبلیِ همین مدل سازگارند.
- **بار اول اجرا** مدل دانلود می‌شود (حدوداً 1.3GB در cache) و بعد از آن اجراهای بعدی حتی آفلاین هم کار می‌کنند.
- سرعت روی CPU: برای کل یک کتاب معمولاً در حد چند تا ۳۰ دقیقه (بسته به CPU). برای تست، اول `END_PAGE = 10` بگذارید.
- کلید API لازم ندارد؛ در `.env` فقط `DATABASE_URL2` لازم است.

**جایگزین رایگان API** (اگر لپ‌تاپ برای اجرای محلی مناسب نبود):

| سرویس | مدل | ابعاد | سهمیه‌ی رایگان (تأییدشده) | نکته |
|---|---|---|---|---|
| Cohere — Trial key | `embed-english-v3.0` | 1024 | 1000 فراخوانی در ماه، 2000 ورودی در دقیقه | context این مدل 512 توکن است → باید `truncate=END` ست شود |

> 📌 برای کل یک کتاب (~۳۰۰-۵۰۰ چانک با batch=32) حدوداً ۱۰-۱۷ فراخوانی لازم است — خیلی داخل سهمیه‌ی رایگان Cohere. اگر این گزینه را خواستی بگو تا نوت‌بوکش را بسازم.
>
> ⚠️ قاعده: هر مدل دیگری انتخاب شود، بردارهایش با بردارهای مدل فعلی قابل مقایسه نیستند → باید کل جدول drop و همه‌ی منابع دوباره ingest شوند، و در query هم دقیقاً همان مدل استفاده شود. الان که جدول خالی است بهترین لحظه برای انتخاب نهایی است.

In [ ]:
import psycopg
from pgvector.psycopg import register_vector

with psycopg.connect(DB_URI) as conn:
    with conn.cursor() as cur:
        cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
        conn.commit()

    register_vector(conn)

    with conn.cursor() as cur:
        create_table_query = """
        DROP TABLE IF EXISTS book_chunks;
        CREATE TABLE IF NOT EXISTS book_chunks (
            id SERIAL PRIMARY KEY,
            chunk_id VARCHAR(100) UNIQUE NOT NULL,
            source_title VARCHAR(255) NOT NULL,
            page_number INT NOT NULL,
            content TEXT NOT NULL,
            token_count INT NOT NULL,
            embedding vector(1024)
        );

        CREATE INDEX IF NOT EXISTS book_chunks_embedding_idx
        ON book_chunks USING hnsw (embedding vector_cosine_ops);
        """
        cur.execute(create_table_query)
        conn.commit()

print("done")


In [ ]:
# ---------- مسیر PDF روی لپ‌تاپ ----------
import os

# ویندوز: از رشته‌ی خام استفاده کنید، مثلاً:  PDF_PATH = r"C:\Users\You\Desktop\book.pdf"
PDF_PATH = "computer-networking-a-top-down-approach-8th-edition.pdf"  # <-- مسیر فایل PDF را اینجا ست کنید

if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(
        f"❌ فایل PDF پیدا نشد: {PDF_PATH}\n"
        f"   پوشه‌ی جاری: {os.getcwd()}\n"
        "   مسیر PDF_PATH را در همین سلول اصلاح کنید (این مقدار در سلول‌های بعدی هم استفاده می‌شود)."
    )
print(f"✅ PDF found: {os.path.abspath(PDF_PATH)}")

In [ ]:
import pypdf

PDF_FILE_PATH = PDF_PATH  # از سلول قبل می‌آید — دیگر نیازی به Google Drive نیست

reader = pypdf.PdfReader(PDF_FILE_PATH)
print(f"book pages: {len(reader.pages)}")

sample_page = reader.pages[5]
print("sample of extracted sentece")
print(sample_page.extract_text()[:500])

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

In [ ]:
import os
import numpy as np
import tiktoken
from pypdf import PdfReader
import psycopg
from pgvector.psycopg import register_vector
from sentence_transformers import SentenceTransformer

# ================= Configuration =================
# DB_URI در سلول اول از .env خوانده شده است.
# PDF_PATH در سلول مسیر PDF ست شده است.
SOURCE_TITLE = "AI Engineering"   # ← عنوان واقعی منبع را مطابق PDF خودت ست کن (برای فیلتر کردن در retrieval)

MODEL_NAME = "BAAI/bge-large-en-v1.5"   # مدل رایگان متن‌باز — 1024 dimensions
DEVICE = "cpu"            # اگر GPU دارید: "cuda"
ENCODE_BATCH = 16         # batch size برای encode روی CPU (روی CPU ضعیف‌تر: 8)

CHUNK_SIZE = 500       # Target tokens per chunk
CHUNK_OVERLAP = 50     # Overlap tokens between chunks
BATCH_SIZE = 50        # Database batch insert size

# For full book: START_PAGE = 0, END_PAGE = None
START_PAGE = 0
END_PAGE = None       # Set to None to automatically process all pages
# =================================================


def get_token_encoder():
    return tiktoken.get_encoding("cl100k_base")


def extract_chunks_from_pdf(pdf_path, start_page=0, end_page=None):
    print(f"📖 Loading PDF from: {pdf_path}")
    reader = PdfReader(pdf_path)
    total_pages = len(reader.pages)
    print(f"📑 Total pages detected: {total_pages}")

    if end_page is None or end_page > total_pages:
        end_page = total_pages

    tokenizer = get_token_encoder()
    chunks = []
    chunk_counter = 1

    print(f"⚙️ Chunking pages {start_page} to {end_page}...")

    for page_idx in range(start_page, end_page):
        page = reader.pages[page_idx]
        text = page.extract_text() or ""

        # Clean basic whitespace
        text = " ".join(text.split())
        if not text:
            continue

        tokens = tokenizer.encode(text)
        page_num = page_idx + 1  # 1-based page number for human readability

        # Sliding window chunking
        for i in range(0, len(tokens), CHUNK_SIZE - CHUNK_OVERLAP):
            chunk_tokens = tokens[i:i + CHUNK_SIZE]
            chunk_text = tokenizer.decode(chunk_tokens)

            chunk_id = f"chunk_p{page_num}_{chunk_counter}"
            chunk_counter += 1

            chunks.append({
                "chunk_id": chunk_id,
                "page_number": page_num,
                "content": chunk_text,
                "token_count": len(chunk_tokens)
            })

    print(f"✅ Generated {len(chunks)} chunks in total.")
    return chunks


_model = None


def get_model():
    global _model
    if _model is None:
        print(f"⬇️ Loading model: {MODEL_NAME} (بار اول ~1.3GB دانلود و در cache ذخیره می‌شود)")
        _model = SentenceTransformer(MODEL_NAME, device=DEVICE)
    return _model


def embed_texts(texts):
    """
    Embedding محلی با sentence-transformers — کاملاً رایگان، بدون API.
    خروجی: آرایه numpy به شکل (len(texts), 1024) — نرمال‌شده L2 (normalize_embeddings=True)
    """
    model = get_model()
    emb = model.encode(
        texts,
        batch_size=ENCODE_BATCH,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    )
    embeddings = np.asarray(emb, dtype=np.float32)
    if np.isnan(embeddings).any() or embeddings.shape != (len(texts), 1024):
        raise RuntimeError(f"Unexpected embedding shape/dtype: {embeddings.shape}")
    return embeddings


def main():
    # 1. Chunk the PDF
    chunks = extract_chunks_from_pdf(PDF_PATH, START_PAGE, END_PAGE)
    if not chunks:
        print("⚠️ No chunks found. Please verify PDF path and page range.")
        return

    # 2. Generate Embeddings locally — بدون هیچ کلید و API
    texts_to_embed = [chunk["content"] for chunk in chunks]
    embeddings = embed_texts(texts_to_embed)

    # 3. Connect to Supabase & Batch Insert
    print("🔌 Connecting to Supabase database...")
    insert_query = """
        INSERT INTO book_chunks (
            chunk_id,
            source_title,
            page_number,
            content,
            token_count,
            embedding
        ) VALUES (%s, %s, %s, %s, %s, %s);
    """

    with psycopg.connect(DB_URI) as conn:
        register_vector(conn)
        with conn.cursor() as cur:
            print("💾 Inserting chunks into Supabase table 'book_chunks'...")

            records_batch = []
            for idx, (chunk, emb) in enumerate(zip(chunks, embeddings)):
                records_batch.append((
                    chunk["chunk_id"],
                    SOURCE_TITLE,
                    chunk["page_number"],
                    chunk["content"],
                    chunk["token_count"],
                    emb
                ))

                # Commit in batches to optimize network I/O
                if len(records_batch) >= BATCH_SIZE:
                    cur.executemany(insert_query, records_batch)
                    conn.commit()
                    records_batch.clear()
                    print(f"📦 Committed {idx + 1}/{len(chunks)} chunks...")

            # Insert any remaining records
            if records_batch:
                cur.executemany(insert_query, records_batch)
                conn.commit()

    print("🎉 Ingestion complete! All chunks and vector embeddings successfully saved to Supabase.")


if __name__ == "__main__":
    main()